This is part 17 of a tutorial series. We recommend following them in order, starting with [Part 0: Welcome to `musica`](0.%20Welcome%20to%20MUSICA.ipynb).

# MIEM NOx Emissions Box Model

This tutorial demonstrates `musica.miem` -- the Python interface to MIEM, MUSICA's emissions model -- driving a real surface NO emission flux into TS1's tropospheric chemistry, inside a single well-mixed box.

We deliberately build on [Part 11: TS1 Box Model](11.%20ts1_box_model.ipynb)'s mechanism rather than the [Chapman Cycle](10.%20chapman.ipynb)'s: Chapman only has oxygen chemistry (N2, O2, O, O1D, O3) with no nitrogen species at all, whereas TS1 already includes a full, real, validated NOx family (NO, NO2, NO3, N2O5, HNO3, PAN, ...). Reusing TS1's existing chemistry means this tutorial can focus entirely on the new piece -- driving an emission source with `miem` -- without inventing new reaction rate constants.

The moving parts:
1. **Chemistry**: TS1's real mechanism, run as a single grid cell (a well-mixed "box") instead of TS1's usual 10-level vertical column.
2. **Emissions**: a `musica.miem.Emissions` module built from an in-code `EmissionsConfig`, reading a small synthetic surface NO inventory.
3. **Coupling**: each time step, `emissions.run(...)` returns a surface flux `[kg m-2 s-1]`; we convert that into a concentration increment `[mol m-3]` for the box using its height and NO's molecular weight, and add it to the state before solving.

## 1. Imports

In [ ]:
import json
import tempfile
from datetime import datetime, time, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import netCDF4
import pandas as pd
import pvlib
import ussa1976
import xarray as xr

import musica
from musica.mechanism_configuration import (
    EmissionsConfig,
    Inventory,
    Mechanism,
    Regridding,
    RegriddingType,
    SourceDescriptor,
    SourceMode,
    SourceType,
    SpeciesMap,
    SpeciesMapping,
    TemporalInterpolation,
    VerticalInjection,
    parse,
)
from musica.micm.solver_result import SolverState
from musica.miem import Emissions
from musica.tuvx import vTS1
from musica.utils import find_config_path

SECONDS_PER_HOUR = 3600
NO_MOLECULAR_WEIGHT = 0.030006  # kg mol-1

## 2. Location, Simulation Time, and Box Geometry

Same location as the Chapman/TS1 tutorials (Boulder, CO). We run a single grid cell -- index 1 of TS1's vertical edges, skipping ground level 0 -- treated as one well-mixed box of a given height, used below to convert surface flux into a concentration tendency.

In [ ]:
boulder = (40.01879858223568, -105.27492413846649)  # (latitude, longitude)
boulder_tz = ZoneInfo("America/Denver")

today_local = datetime.now(boulder_tz).date()
noon_local = datetime.combine(today_local, time(7, 30), tzinfo=boulder_tz)
sim_time = (noon_local - timedelta(hours=1)).astimezone(ZoneInfo("UTC"))

grid_cell_index = 1  # skip ground-level index 0, matching the chapman/ts1 tutorials
num_grid_cells = 1

# Representative surface NOx flux [kg m-2 s-1] for the synthetic inventory below --
# on the order of a moderately busy urban surface source.
NO_SURFACE_FLUX = 2.0e-9

# Height of the well-mixed box, used to convert surface flux into a concentration tendency.
BOX_HEIGHT_M = 100.0

print(f"Simulation start (UTC): {sim_time}")

## 3. TS1 Mechanism and Solver

We reuse TS1's real mechanism and initial conditions as-is (`configs/v1/ts1/`), but create the solver's state with a single grid cell instead of TS1's usual 10-level vertical column.

In [ ]:
mechanism = parse(find_config_path("v1", "ts1", "ts1.json"))
solver = musica.MICM(mechanism=mechanism, solver_type=musica.SolverType.rosenbrock_standard_order)
state = solver.create_state(num_grid_cells)

## 4. Photolysis Rates from TUV-x\n\nSame TS1 alias-mapping approach as [Part 11](11.%20ts1_box_model.ipynb), restricted to our single grid cell.

In [ ]:
tuvx = vTS1.get_tuvx_calculator()

solpos = pvlib.solarposition.get_solarposition(time=sim_time, latitude=boulder[0], longitude=boulder[1])
sza = solpos['zenith'].item()
tuv_rates = tuvx.run(sza=np.deg2rad(sza), earth_sun_distance=1.0)

tuv_path = find_config_path("tuvx", "ts1_tsmlt.json")
with open(tuv_path, 'r') as f:
    tuv_config = json.load(f)
alias_mappings = tuv_config.get('__CAM options', {}).get('aliasing', {}).get('pairs', {})

photolysis_rate_constants = {}
for mapping in alias_mappings:
    label = mapping['to']
    scale = mapping.get("scale by", 1)
    tuv_label = mapping['from']
    rate = tuv_rates.sel(reaction=tuv_label).photolysis_rate_constants.values * scale
    photolysis_rate_constants[f'USER.{label}'] = [rate[grid_cell_index]]

vertical_edge = tuv_rates.vertical_edge[grid_cell_index].item()
print(f"Grid cell altitude: {vertical_edge:.2f} km")

## 5. Initial Conditions\n\nReused directly from TS1's `initial_conditions.csv` -- including its real NO, NO2, and O3 starting concentrations.

In [ ]:
conditions = pd.read_csv(
    find_config_path("v1", "ts1", "initial_conditions.csv"),
    sep=',', names=['parameter', 'value1', 'value2'],
    dtype={'parameter': str, 'value1': float, 'value2': float})

initial_concentrations = conditions[conditions['parameter'].str.contains('CONC')].copy()
initial_concentrations['parameter'] = initial_concentrations['parameter'].str.replace('CONC.', '', regex=False)

surface_reactions = conditions[conditions['parameter'].str.contains('SURF')]
user_defined_conditions = conditions[conditions['parameter'].str.contains('USER')]

concentration_dict = {row['parameter']: [row['value1']] for _, row in initial_concentrations.iterrows()}

user_defined_dict = {row['parameter']: [row['value1']] for _, row in user_defined_conditions.iterrows()}
for _, row in surface_reactions.iterrows():
    user_defined_dict[f"{row['parameter']}.effective radius [m]"] = [row['value1']]
    user_defined_dict[f"{row['parameter']}.particle number concentration [# m-3]"] = [row['value2']]
user_defined_dict.update(photolysis_rate_constants)

environmental_conditions = ussa1976.compute(z=np.array([vertical_edge * 1000]), variables=["t", "p"])
temperature = environmental_conditions['t'].values
pressure = environmental_conditions['p'].values

state.set_conditions(temperature, pressure)
state.set_concentrations(concentration_dict)
state.set_user_defined_rate_parameters(user_defined_dict)

print(f"Initial NO:  {concentration_dict['NO'][0]:.3e} mol m-3")
print(f"Initial NO2: {concentration_dict['NO2'][0]:.3e} mol m-3")
print(f"Initial O3:  {concentration_dict['O3'][0]:.3e} mol m-3")

## 6. Build the Emissions Source with `musica.miem`

This is the new piece. `EmissionsConfig` describes *where* the NO flux comes from (an inventory file, a species map resolving the inventory's variable name to the mechanism species `NO`, and a source tying them together) -- built here directly in Python rather than parsed from a file, exactly like the chemistry `Mechanism` above could be.

The emissions module needs a real (here, synthetic) NetCDF inventory file. We write a minimal single-snapshot file using the `uptempo` convention: just an `nCells` dimension and a 1D flux variable named after the inventory species -- no time axis is required for a single-snapshot file.

In [ ]:
def write_synthetic_no_inventory(path, flux_value=NO_SURFACE_FLUX):
    ds = netCDF4.Dataset(str(path), "w", format="NETCDF4")
    try:
        ds.createDimension("nCells", 1)
        flux = ds.createVariable("no_surface", "f8", ("nCells",))
        flux[:] = np.array([flux_value])
    finally:
        ds.close()


def get_emissions(nc_path):
    emissions_config = EmissionsConfig(
        inventories=[
            Inventory(name="no inventory", directory="", file_pattern=str(nc_path), convention="uptempo"),
        ],
        species_maps=[
            SpeciesMap(
                name="no map",
                mappings=[SpeciesMapping(inventory_species="no_surface", mechanism_species="NO", scaling_factor=1.0)],
            ),
        ],
        regridding=Regridding(type=RegriddingType.None_),
        sources=[
            SourceDescriptor(
                name="no source",
                mode=SourceMode.Offline,
                type=SourceType.Anthropogenic,
                inventory="no inventory",
                species_map="no map",
                temporal_interpolation=TemporalInterpolation.None_,
                vertical_injection=VerticalInjection.Surface,
                category=0,
                hierarchy=1,
                scaling_factor=1.0,
                sector="anthropogenic",
            ),
        ],
    )
    emissions_mechanism = Mechanism(name="no emissions", emissions=emissions_config)
    return Emissions(mechanism=emissions_mechanism, n_cells=1, n_vert_levels=1)


tmp_dir = tempfile.mkdtemp()
nc_path = Path(tmp_dir) / "no_surface.nc"
write_synthetic_no_inventory(nc_path)
emissions = get_emissions(nc_path)

print(f"Emissions species: {list(emissions.species_names)}")

## 7. Run the Simulation

Each time step:
1. Call `emissions.run(epoch_seconds, dt_seconds)` -- returns surface flux `[kg m-2 s-1]` as a `(num_species, n_cells)` numpy array.
2. Convert that flux into a concentration increment for the box: `Δ[NO] = flux * dt / (box_height * molecular_weight)`.
3. Add the increment to the current NO concentration before solving the chemistry forward.

In [ ]:
sim_times = [sim_time]
concentrations = [state.get_concentrations()]
no_flux_history = [0.0]
time_step = 30  # seconds
simulation_length = 3 * SECONDS_PER_HOUR
current_time = 0
last_printed_percent = -5

while current_time < simulation_length:
    flux = emissions.run(sim_time.timestamp(), time_step)
    no_index = list(emissions.species_names).index("NO")
    no_surface_flux = flux[no_index, 0]  # kg m-2 s-1, single cell

    delta_no_mol_m3 = no_surface_flux * time_step / (BOX_HEIGHT_M * NO_MOLECULAR_WEIGHT)

    current_concentrations = state.get_concentrations()
    current_concentrations["NO"] = [current_concentrations["NO"][0] + delta_no_mol_m3]
    state.set_concentrations(current_concentrations)

    elapsed = 0
    while elapsed < time_step:
        remaining_time = time_step - elapsed
        result = solver.solve(state, remaining_time)
        elapsed += result.stats.final_time
        current_time += result.stats.final_time
        if result.state != SolverState.Converged:
            print(f"Solver state: {result.state}, time: {current_time}")

    current_percent = (current_time / simulation_length) * 100
    if int(current_percent // 5) * 5 > last_printed_percent:
        last_printed_percent = int(current_percent // 5) * 5
        print(f"Simulation progress: {last_printed_percent}%")

    sim_time += timedelta(seconds=time_step)
    sim_times.append(sim_time)
    concentrations.append(state.get_concentrations())
    no_flux_history.append(no_surface_flux)

## 8. Build the xarray Dataset

In [ ]:
data_vars = {}
species_ordering = state.get_species_ordering()
for species in species_ordering:
    data_vars[species] = (["time"], [c[species][0] for c in concentrations])
data_vars["no_surface_flux"] = (["time"], no_flux_history, {"units": "kg m-2 s-1"})

coords = {"time": np.array([int(t.timestamp()) for t in sim_times], dtype="datetime64[s]")}
ds = xr.Dataset(data_vars, coords=coords)
ds

## 9. Plot Results\n\nNO tracks the emitted flux directly; NO2 and O3 respond through TS1's real NOx-ozone chemistry.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
time_hours = (ds['time'] - ds['time'].isel(time=0)) / np.timedelta64(1, 'h')

axes[0, 0].plot(time_hours, ds['NO'])
axes[0, 0].set_title('NO')
axes[0, 1].plot(time_hours, ds['NO2'])
axes[0, 1].set_title('NO2')
axes[1, 0].plot(time_hours, ds['O3'])
axes[1, 0].set_title('O3')
axes[1, 1].plot(time_hours, ds['no_surface_flux'])
axes[1, 1].set_title('NO surface flux (miem)')
axes[1, 1].set_ylabel('Flux [kg m-2 s-1]')

for _ax in axes.flat[:3]:
    _ax.set_ylabel('Concentration [mol m-3]')
for _ax in axes.flat:
    _ax.grid(True, alpha=0.5)
    _ax.spines[:].set_visible(False)
    _ax.tick_params(width=0)
    _ax.set_xlim(0, simulation_length / SECONDS_PER_HOUR)
    _ax.set_xlabel('Time [hours]')

fig.tight_layout()